In [2]:
import pandas as pd
import numpy as np

In [3]:
np.random.seed(1)

In [9]:
subjects = np.repeat(["S1", "S2", "S3"], 6)
conditions = np.tile(np.repeat(["A", "B"], 3), 3)
trials = np.tile([1, 2, 3], 6)
rt = np.random.normal(loc=500, scale=50, size=18)

df = pd.DataFrame({
    "Subject": subjects,
    "Condition": conditions,
    "Trial": trials,
    "RT": rt
})

> pivot() performs a pure reshaping operation. It does not aggregate.

Note: the combination of (index, columns) must uniquely identify a single value. If duplicates exist, pivot() raises an error.

In [10]:
# df.pivot(index="Subject", columns="Condition", values="RT") fails because multiple trials per Subject × Condition exist.

Note: pivot() is only appropriate when data are already aggregated or uniquely indexed.

> pivot_table() is a generalization of pivot(). It handles duplicates by aggregating.

Equivalent to:
> groupby(...) + aggregation + unstack()

Note: pivot_table() is syntactic convenience for grouped aggregation.

In [15]:
# Mean rt per subject per condition:
table1 = df.pivot_table(
    index="Subject",
    columns="Condition",
    values="RT",
    aggfunc="mean" # Can be string ("mean", "sum"), function (np.mean), or list of functions.
    # margins=True: adds row/column totals.
    # fill_value: replaces NaN cells.
)
print(table1)

# Using multiple aggfuncs produces multi-index columns:

table2 = df.pivot_table(
    index="Subject",
    columns="Condition",
    values="RT",
    aggfunc=["mean","std"]
)
print(table2)

Condition           A           B
Subject                          
S1         516.813156  497.701734
S2         484.662372  524.590186
S3         498.369305  499.270638
                 mean                    std           
Condition           A           B          A          B
Subject                                                
S1         516.813156  497.701734  39.895287  27.087060
S2         484.662372  524.590186  16.667547  42.734340
S3         498.369305  499.270638  46.243674  43.099025


> melt() converts wide to long (columns become rows).

In [20]:
long_df = pd.melt(
    table1.reset_index(),
    id_vars="Subject",
    value_vars=["A","B"],
    var_name="Condition",
    value_name="Mean_RT"
)
print(long_df)

  Subject Condition     Mean_RT
0      S1         A  516.813156
1      S2         A  484.662372
2      S3         A  498.369305
3      S1         B  497.701734
4      S2         B  524.590186
5      S3         B  499.270638


> stack() moves columns to inner row index.

> unstack() moves row index level → columns.

In [21]:
grouped = df.groupby(["Subject","Condition"])["RT"].mean()
print("Grouped:\n", grouped)

wide = grouped.unstack() # This produces the same result as pivot_table (df.pivot_table(...)).
print("\nWide :\n", wide)

# stack() does the inverse:
long_again = wide.stack()
print("\nLong again:\n", long_again)

Grouped:
 Subject  Condition
S1       A            516.813156
         B            497.701734
S2       A            484.662372
         B            524.590186
S3       A            498.369305
         B            499.270638
Name: RT, dtype: float64

Wide :
 Condition           A           B
Subject                          
S1         516.813156  497.701734
S2         484.662372  524.590186
S3         498.369305  499.270638

Long again:
 Subject  Condition
S1       A            516.813156
         B            497.701734
S2       A            484.662372
         B            524.590186
S3       A            498.369305
         B            499.270638
dtype: float64


Note: for plotting, long format is usually preferred.

#### Long vs Wide Format in Research

- Long (tidy) format:
    - Each row = one observation; each column = one variable.
    - Advantages: fits linear/mixed models, easy to add covariates, scalable for hierarchical designs.
    - Used by most modern statistical frameworks (Python, R).

- Wide format:
    - Each row = one subject; repeated measures are columns.
    - Advantages: convenient for descriptive summaries, easy to inspect subject-level data.
    - Limitations: not scalable for many conditions, harder to include time-varying covariates, less compatible with mixed models.

Note: Long format is preferred for analysis; wide format is mostly for visualization or summary tables.